In [ ]:

import os
import yaml
import pandas as pd
from pprint import pprint
from tqdm import tqdm

def load_yaml(path):
    """Load a YAML file into a Python dict."""
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def extract_jobs(data):
    """Return the 'jobs' section from the workflow dict,
       treating None (or any non-dict) as “no jobs”."""
    if not isinstance(data, dict):
        return {}
    return data.get('jobs', {})


def extract_steps(job_spec):
    """Map step name to its spec, defaulting any missing/null name to <step_{idx}>."""
    if not isinstance(job_spec, dict):
        return {}

    steps = job_spec.get('steps', [])
    mapping = {}
    for idx, step in enumerate(steps):
        raw = step.get('name')
        if not isinstance(raw, str) or not raw.strip():
            name = f"<step_{idx}>"
        else:
            name = raw
        mapping[name] = step
    return mapping


def compare_jobs(jobs1, jobs2, file1_label='file1', file2_label='file2'):
    """Produce a list of diffs at job/step/property level."""
    diffs = []
    all_jobs = set(jobs1) | set(jobs2)
    for job in sorted(all_jobs):
        j1, j2 = jobs1.get(job), jobs2.get(job)
        if j1 is None:
            diffs.append({
                'prev_file': file1_label,
                'current_file': file2_label,
                'job': job, 'step': None,
                'from': None, 'to': j2,
                'change': 'job added'
            })
        elif j2 is None:
            diffs.append({
                'prev_file': file1_label,
                'current_file': file2_label,
                'job': job, 'step': None,
                'from': j1, 'to': None,
                'change': 'job removed'
            })
        else:
            s1_map = extract_steps(j1)
            s2_map = extract_steps(j2)
            all_steps = set(s1_map) | set(s2_map)
            for step in sorted(all_steps):
                s1, s2 = s1_map.get(step), s2_map.get(step)
                if s1 is None:
                    diffs.append({
                        'prev_file': file1_label,
                        'current_file': file2_label,
                        'job': job, 'step': step,
                        'from': None, 'to': s2,
                        'change': 'step added'
                    })
                elif s2 is None:
                    diffs.append({
                        'prev_file': file1_label,
                        'current_file': file2_label,
                        'job': job, 'step': step,
                        'from': s1, 'to': None,
                        'change': 'step removed'
                    })
                else:
                    keys = set(s1) | set(s2)
                    for key in sorted(keys):
                        v1, v2 = s1.get(key), s2.get(key)
                        if v1 != v2:
                            diffs.append({
                                'prev_file': file1_label,
                                'current_file': file2_label,
                                'job': job, 'step': step,
                                'from': {key: v1}, 'to': {key: v2},
                                'change': f"{key} changed"
                            })
    return diffs

import glob

BASE_DIR = ''

csv_files = glob.glob(os.path.join(BASE_DIR, '*.csv'))
meta = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

meta['committed_date'] = pd.to_datetime(meta['committed_date'], unit='s')
meta.rename(columns={
    'file_hash': 'file_1_hash',
    'previous_file_hash': 'file_2_hash'
}, inplace=True)

meta.sort_values(['repository', 'committed_date', 'file_path'], inplace=True)

WORKFLOW_BASE_DIR = BASE_DIR  # now points to the parent folder containing all *_workflows subfolders

all_diffs = []
for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Diffing commits"):
    h_curr = str(row['file_1_hash']).strip()
    h_prev = str(row['file_2_hash']).strip()

    if not h_prev or pd.isna(row['file_2_hash']):
        no_prev = {
            'prev_file': None,
            'current_file': f"{h_curr} (curr)",
            'job': None,
            'step': None,
            'from': None,
            'to': None,
            'change': 'no prev file',
            'repository':      row['repository'],
            'commit_hash':     row['commit_hash'],
            'committed_date':  row['committed_date'],
            'file_path':       row['file_path'],
            'git_change_type': row['git_change_type']
        }
        all_diffs.append(no_prev)
        continue

    #  Locate the YAML files by searching recursively in all *_workflows subdirectories
    curr_matches = glob.glob(os.path.join(WORKFLOW_BASE_DIR, '**', h_curr), recursive=True)
    prev_matches = glob.glob(os.path.join(WORKFLOW_BASE_DIR, '**', h_prev), recursive=True)

    # If either file is missing, skip this row
    if not curr_matches or not prev_matches:
        continue

    f_curr = curr_matches[0]
    f_prev = prev_matches[0]

    # 3. Load and diff as before
    try:
        w_curr = load_yaml(f_curr)
        w_prev = load_yaml(f_prev)
    except Exception:
        continue
    
    folder_prev = os.path.basename(os.path.dirname(f_prev))
    folder_curr = os.path.basename(os.path.dirname(f_curr))

#debug
    if w_prev is None:
        print(f"WARNING: {f_prev} loaded as None (from folder: {folder_prev})")
    if w_curr is None:
        print(f"WARNING: {f_curr} loaded as None (from folder: {folder_curr})")
        
    diffs = compare_jobs(
        extract_jobs(w_prev),
        extract_jobs(w_curr),
        file1_label=f"{h_prev} (prev)",
        file2_label=f"{h_curr} (curr)"
    )

    #  Enrich and collect diffs
    for d in diffs:
        d.update({
            'repository':      row['repository'],
            'commit_hash':     row['commit_hash'],
            'committed_date':  row['committed_date'],
            'file_path':       row['file_path'],
            'git_change_type': row['git_change_type']
        })
        all_diffs.append(d)

diffs_df = pd.DataFrame(all_diffs)
cols = [
    'repository','commit_hash','committed_date','file_path','git_change_type',
    'prev_file','current_file','job','step','from','to','change'
]
diffs_df = diffs_df[cols]
diffs_df.head()

Diffing commits:  17%|█▋        | 3210/18473 [02:55<08:43, 29.16it/s] 

Diffing commits:  17%|█▋        | 3231/18473 [02:56<11:01, 23.04it/s]

Diffing commits:  41%|████      | 7496/18473 [06:32<06:36, 27.66it/s]

Diffing commits:  47%|████▋     | 8705/18473 [07:38<05:29, 29.66it/s]

Diffing commits:  49%|████▉     | 9028/18473 [07:51<06:43, 23.38it/s]

Diffing commits:  99%|█████████▊| 18202/18473 [15:00<00:11, 23.71it/s]

Diffing commits:  99%|█████████▊| 18208/18473 [15:00<00:10, 24.22it/s]

Diffing commits: 100%|██████████| 18473/18473 [15:13<00:00, 20.23it/s]


,repository,commit_hash,committed_date,file_path,git_change_type,prev_file,current_file,job,step,from,to,change
0,AMII,ad017a65b8a559de4aa8879a35a834d382d446a4,2020-10-12 10:18:00,.github/workflows/build.yml,A,None,5eec24006084f57e9d0ac4527eda73fb0ddbc97c1d2a73...,None,None,None,None,no prev file
1,AMII,ad017a65b8a559de4aa8879a35a834d382d446a4,2020-10-12 10:18:00,.github/workflows/release.yml,A,None,db45597abcae2abee24c94914182ea147eaac6b1121703...,None,None,None,None,no prev file
2,AMII,ad017a65b8a559de4aa8879a35a834d382d446a4,2020-10-12 10:18:00,.github/workflows/template-cleanup.yml,A,None,a33733d2da31d156db4688cbca2228429942239a4dba5b...,None,None,None,None,no prev file
3,AMII,0c2c0e278104d383e4af7db633d7d858153868c3,2020-10-14 10:29:57,.github/workflows/build.yml,M,5eec24006084f57e9d0ac4527eda73fb0ddbc97c1d2a73...,843d8b03dc40ec8b2dfe936ef7192b19c60bda59a16cf3...,build,Export Properties,"{'name': 'Export Properties', 'id': 'propertie...",None,step removed
4,AMII,0c2c0e278104d383e4af7db633d7d858153868c3,2020-10-14 10:29:57,.github/workflows/build.yml,M,5eec24006084f57e9d0ac4527eda73fb0ddbc97c1d2a73...,843d8b03dc40ec8b2dfe936ef7192b19c60bda59a16cf3...,build,Set environment variables,None,"{'name': 'Set environment variables', 'id': 'p...",step added
